In [1]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from app.logger import *
import json5,json
import fitz #type: ignore

from app.amc.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

In [12]:
amc_id = '20_0'
path = r"20_31-Dec-25_FS.pdf"
config = get_config("2025",amc_id)
regex = get_regex("2025")

object = GROWW(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
print("done")
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2025\20_0_AMC.json5
done


In [11]:
import pandas as pd
import json

path =r"C:\Users\kaustubh.keny\Downloads\search (4).json"

with open(path,"r") as f:
    
    data = json.load(f)
    
collect = []
imp_data = data["hits"]["hits"]
# # len(imp_data)
# df = pd.DataFrame(imp_data)
# df.to_csv("check.csv")
for item in imp_data:
    
    entry = item["_source"]
    collect.append(entry)
    
df = pd.DataFrame(collect)
df.to_csv("check4.csv")

In [ ]:
# len(title)
title

In [16]:
config = get_config("2025",amc_id)
regex = get_regex("2025")
object = GROWW(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\mywork-repo\config\2025\20_0_AMC.json5


In [17]:
save_path = os.path.join(object.JSONPATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\20_31-Dec-25_FS.json


In [ ]:
pattern = "^(?!.*Benchmark)(.+)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith("manager"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

In [ ]:
#bajaj
# def _update_bench_data(self,main_key:str,data):
#     data = " ".join(data) if isinstance(data, list) else data
#     matches = re.findall(self.REGEX["benchmark"],data,re.IGNORECASE)
#     return {"benchmark_index":matches[0] if matches else ""}

# def _update_date_data(self,main_key:str,data):
#     data = " ".join(data) if isinstance(data, list) else data
#     matches = re.findall(self.REGEX["date"],data,re.IGNORECASE)
#     return {"scheme_launch_date":matches[0] if matches else ""}
 

#generic
def _update_benchmark_data(self,main_key:str,bench_data):
    bench_data = " ".join(bench_data) if isinstance(bench_data,list) else bench_data
    bench_data = re.sub(self.REGEX["escape"],"",bench_data,re.IGNORECASE)
    if match:=re.match(self.REGEX["benchmark"],bench_data,re.IGNORECASE):
        return {"benchmark_index":match[0]}
    return {main_key:bench_data}
   

def _update_date_data(self,main_key:str,data):
    date_data = " ".join(data) if isinstance(data, list) else data
    if match := re.findall(self.REGEX["date"],date_data,re.IGNORECASE):
        return {"scheme_launch_date":match[0]}
    return {main_key:date_data}

   
#canara
# def _update_benchmark_data(self,main_key:str,bench_data):
#     bench_data = " ".join(bench_data) if isinstance(bench_data,list) else bench_data
#     bench_data = re.sub(self.REGEX["escape"],"",bench_data,re.IGNORECASE)
#     if match:=re.match(self.REGEX["benchmark"],bench_data,re.IGNORECASE):
#         return {main_key:match[0]}
#     return {main_key:bench_data}

#dsp
# def _update_benchmark_data(self,main_key:str,bench_data):
#     bench_data = " ".join(bench_data) if isinstance(bench_data,list) else bench_data
#     bench_data = re.sub(self.REGEX["escape"],"",bench_data,re.IGNORECASE)
#     if match:=re.match(self.REGEX["benchmark"],bench_data,re.IGNORECASE):
#         return {main_key:match[0]}
#     return {main_key:bench_data}

# def _update_date_data(self,main_key:str,data):
#     data = " ".join(data) if isinstance(data, list) else data
#     matches = re.findall(self.REGEX["date"],data,re.IGNORECASE)
#     return {"scheme_launch_date":matches[0] if matches else ""}

#hdfc
# def _update_date_data(self,main_key:str,data):
#     if matches:=re.findall(self.REGEX["date"],data, re.IGNORECASE):
#         return {main_key:matches[0]}
def _update_benchmark_data(self,main_key:str,data):
    data=re.sub(self.REGEX["benchmark"],"", data, re.IGNORECASE).strip()
    if matches:= re.findall(self.REGEX["benchmark2"],data,re.IGNORECASE):
        return {main_key:matches[0]}
    return {main_key:data}

#navi
# def _extract_benchmark_data(self,main_key:str,data:str,pattern:str):
#     bench_data = f"{main_key} {data}"
#     bench_data = re.sub(self.REGEX["escape"],"",bench_data).strip()
#     if matches:=re.findall(self.REGEX[pattern],bench_data, re.IGNORECASE):
#         return {"benchmark_index":matches[0]}
#     return{"benchmark_index":f"{main_key} {data}"}


           
def _extract_bench_data(self,main_key:str,data,pattern:str):
    data = " ".join(data) if isinstance(data,list) else data
    data = re.sub(r"Ni\s*y","Nifty",data, re.IGNORECASE)
    return {main_key:data}
    

# def _update_date_data(self,main_key:str,data):
#     if matches:=re.findall(self.REGEX["date"],data, re.IGNORECASE):
#         return {main_key:matches[0]}

# def _update_date_data(self, main_key:str,data):  # GROWW & Edelweiss
#     date_data = " ".join(data) if isinstance(data,list) else data
#     matches = re.findall(self.REGEX["date"],date_data, re.IGNORECASE)
#     return {"scheme_launch_date": " ".join(matches)}

In [ ]:
import os
import json
import pandas as pd


# ---------------- FIELD CONFIG (converted from your JSON) ---------------- #

FIELD_KEYS = {
    "load_keys": ["entry", "exit"],

    "manager_keys": [
        "name",
        "managing_fund_since",
        "total_exp",
        "qualification"
    ],

    "metric_keys": [
        "alpha",
        "arithmetic_mean_ratio",
        "average_div_yield",
        "average_pb",
        "average_pe",
        "avg_maturity",
        "beta",
        "correlation_ratio",
        "downside_deviation",
        "information_ratio",
        "macaulay",
        "mod_duration",
        "port_turnover_ratio",
        "r_squared_ratio",
        "roe_ratio",
        "sharpe",
        "sortino_ratio",
        "std_dev",
        "tracking_error",
        "treynor_ratio",
        "upside_deviation",
        "ytm"
    ],

    "static_keys": [
        "amc_name",
        "main_scheme_name",
        "mutual_fund_name",
        "benchmark_index",
        "monthly_aaum_date",
        "monthly_aaum_value",
        "scheme_launch_date",
        "min_addl_amt",
        "min_addl_amt_multiple",
        "min_amt",
        "min_amt_multiple",
        "Riskometer_benchmark",
        "Riskometer"
    ],

    "field_location": "field_location"
}


# ---------------- MAIN CONVERTER ---------------- #

def json_to_csv(json_path):

    os.makedirs(output_dir, exist_ok=True)

    static_keys = FIELD_KEYS["static_keys"]
    load_keys = FIELD_KEYS["load_keys"]
    metric_keys = FIELD_KEYS["metric_keys"]
    manager_keys = FIELD_KEYS["manager_keys"]
    field_location = FIELD_KEYS["field_location"]

    EXCLUDE_KEYS = {"Riskometer", "Riskometer_benchmark", "Riskometer_scheme"}

    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    records = doc.get("records", [])

    max_manager = max((len(r["value"].get("fund_manager", [])) for r in records), default=1)

    # headers
    headers = [k for k in static_keys if k not in EXCLUDE_KEYS] + load_keys + metric_keys

    for i in range(1, max_manager + 1):
        headers.extend([f"{k}_{i}" for k in manager_keys])

    headers.append(field_location)

    def flatten(value):

        for bad in EXCLUDE_KEYS:
            value.pop(bad, None)

        row = []

        # static
        for k in static_keys:
            if k in EXCLUDE_KEYS:
                continue

            v = value.get(k, "")
            if isinstance(v, list):
                v = ", ".join(map(str, v))
            row.append(v)

        # load
        entry = ""
        exit_ = ""

        for l in value.get("load", []):
            if l.get("type") == "entry":
                entry = l.get("comment", "")
            elif l.get("type") == "exit":
                exit_ = l.get("comment", "")

        row.extend([entry, exit_])

        # metrics
        metric_map = {m.get("name"): m.get("value") for m in value.get("metrics", [])}
        row.extend([metric_map.get(k, "") for k in metric_keys])

        # managers
        managers = value.get("fund_manager", [])

        for i in range(max_manager):
            if i < len(managers):
                fm = managers[i]
                row.extend([fm.get(k, "") for k in manager_keys])
            else:
                row.extend([""] * len(manager_keys))

        # field location
        fl = value.get("field_location", [{}])
        if isinstance(fl, list) and fl:
            fl_val = json.dumps(fl[0], ensure_ascii=False)
        else:
            fl_val = ""

        row.append(fl_val)

        return row

    rows = [flatten(r["value"]) for r in records]

    for i, r in enumerate(rows[:3]):
        if len(r) != len(headers):
            raise ValueError("Header / row mismatch")

    base = os.path.splitext(os.path.basename(json_path))[0]
  
    pd.DataFrame(rows, columns=headers).to_csv(f"{base}.csv", index=False, encoding="utf-8")

    return f"{base}.csv"


# ---------------- SIMPLE ATTACH FUNCTION ---------------- #

json_file = r"C:\Users\rando\Office Projects\mywork-repo\notebook\fs_sidkim\6_31-Aug-25_FS.json"

def convert_json_file(json_file):
    return json_to_csv(json_file)


convert_json_file(json_file)

'csv_folder\\6_31-Aug-25_FS.csv'

In [11]:
import os, json
import pandas as pd


"Helios Capital Asset Management  (India) Private Limited"

static_keys = [
        "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index", "monthly_aaum_date", 
        "monthly_aaum_value", "scheme_launch_date", "min_addl_amt", "min_addl_amt_multiple", 
        "min_amt", "min_amt_multiple",
    ]
load_keys = ["entry","exit"]
metric_keys = [
    "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe", "avg_maturity",
    "beta", "correlation_ratio", "downside_deviation", "information_ratio", "macaulay",
    "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio", "sharpe", "sortino_ratio",
    "std_dev", "tracking_error", "treynor_ratio", "upside_deviation", "ytm"
]

manager_keys = ["name","managing_fund_since","total_exp","qualification"]

def flatten_to_row(value, max_manager):
    add_value = []

    # Static keys
    for k in static_keys:
        val = value.get(k, "")
        if isinstance(val, list):
            val = ", ".join(val)
        add_value.append(val)

    # Loads
    entry, exit = "", ""
    for l in value.get("load", []):
        if l.get("type") == "entry":
            entry = l.get("comment", "")
        elif l.get("type") == "exit":
            exit = l.get("comment", "")
    add_value.extend([entry, exit])

    # Metrics
    metric_map = {m["name"]: m["value"] for m in value.get("metrics", [])}
    metric = [metric_map.get(k, "") for k in metric_keys]
    add_value.extend(metric)

    # Fund managers (pad to max_manager)
    managers = value.get("fund_manager", [])
    for i in range(max_manager):
        if i < len(managers):
            fm = managers[i]
            add_value.extend([
                fm.get("name", ""),
                fm.get("managing_fund_since", ""),
                fm.get("total_exp", ""),
                fm.get("qualification", "")
            ])
        else:
            # pad with blanks if fewer managers
            add_value.extend(["", "", "", ""])

    return add_value


path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\8_31-Dec-25_FS.json"
with open(path,"r",encoding="utf8") as f:
    df = json.load(f)
sheet_name = df.get("metadata",{}).get("document_name","")
records = df.get("records",[])
max_fund_managers = max(
    (len(record["value"].get("fund_manager", [])) for record in records),
    default=0
)


headers = static_keys + load_keys + metric_keys
for i in range(1, max_fund_managers + 1):
    headers.extend([f"{key}_{i}" for key in manager_keys])

file_name = path.split("\\")[-1].replace(".json","")

rows = [flatten_to_row(record["value"], max_fund_managers) for record in records]
df_out = pd.DataFrame(rows, columns=headers)
df_out.to_csv(f"{file_name}.csv", index=False)



In [ ]:
import pandas as pd
import re, os

path = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\amfii-website\VAHAN_DATA_2025.csv"
df = pd.read_csv(path)
# df.head(10)

months = [f"{m},2025" for m in 
          ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]]

pivot = df.assign(val="1") \
          .pivot_table(index=["STATE","RTO"],
                       columns="DATE",
                       values="val",
                       aggfunc="first") \
          .reindex(columns=months, fill_value="NA")

result = pivot.reset_index()
result.to_csv("validate_2025.csv", index=False)

In [ ]:
def json_to_csv(json_path, output_dir="."):
    keys = {
            "static_keys": [
                "amc_name", "main_scheme_name", "mutual_fund_name", "benchmark_index",
                "monthly_aaum_date", "monthly_aaum_value", "scheme_launch_date",
                "min_addl_amt", "min_addl_amt_multiple", "min_amt", "min_amt_multiple"
            ],
            "load_keys": ["entry", "exit"],
            "metric_keys": [
                "alpha", "arithmetic_mean_ratio", "average_div_yield", "average_pb", "average_pe",
                "avg_maturity", "beta", "correlation_ratio", "downside_deviation", "information_ratio",
                "macaulay", "mod_duration", "port_turnover_ratio", "r_squared_ratio", "roe_ratio",
                "sharpe", "sortino_ratio", "std_dev", "tracking_error", "treynor_ratio",
                "upside_deviation", "ytm"
            ],
            "manager_keys": ["name", "managing_fund_since", "total_exp", "qualification"]
        }
    
    static_keys, load_keys, metric_keys, manager_keys = ( keys["static_keys"], keys["load_keys"], keys["metric_keys"], keys["manager_keys"] )

    with open(json_path, "r", encoding="utf-8") as f:
        doc = json.load(f)

    records = doc.get("records", [])
    max_manager = max((len(r["value"].get("fund_manager", [])) for r in records), default=1)

    # Build headers
    headers = static_keys + metric_keys
    for i in range(1, max_manager + 1):
        headers.extend([f"{k}_{i}" for k in manager_keys])
    headers.extend(load_keys)

    def flatten_to_row(value):
        row = []
        # static
        for k in static_keys:
            v = value.get(k, "")
            if isinstance(v, list):
                v = ", ".join(v)
            row.append(v)
        # metrics
        metric_map = {m.get("name"): m.get("value") for m in value.get("metrics", [])}
        row.extend([metric_map.get(k, "") for k in metric_keys])
        # fund managers
        managers = value.get("fund_manager", [])
        for i in range(max_manager):
            if i < len(managers):
                fm = managers[i]
                row.extend([fm.get(k, "") for k in manager_keys])
            else:
                row.extend([""] * len(manager_keys))
        
        # loads
        entry, exit_ = "", ""
        for l in value.get("load", []):
            if l.get("type") == "entry": entry = l.get("comment", "")
            elif l.get("type") == "exit": exit_ = l.get("comment", "")
        row.extend([entry, exit_])
        
        return row

    rows = [flatten_to_row(r["value"]) for r in records]

    # Timestamped filename
    base = os.path.splitext(os.path.basename(json_path))[0]
    csv_path = os.path.join(output_dir, f"{base}.csv")
    pd.DataFrame(rows, columns=headers).to_csv(csv_path, index=False, encoding="utf-8")
    return csv_path
